# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.

# 6_graph_retrieval.ipynb

PURPOSE:
Build a graph-based retrieval engine over ARIA-Lite V2 graph.

This notebook enables:
- Entity-based querying
- Graph traversal (1-hop and 2-hop)
- Paper retrieval via graph structure
- Structured biomedical reasoning without LLMs

INPUT:
- aria_lite_graph_v2.pkl (from Week 4)

OUTPUT:
- graph_query() function for retrieval

In [1]:
# ============================================================
# SECTION 1 — Install Libraries
# ============================================================

!pip install networkx

In [2]:
# ============================================================
# SECTION 2 — Imports
# ============================================================

from google.colab import drive
import os
import json
import pickle
import networkx as nx
from collections import defaultdict, Counter

In [3]:
# ============================================================
# SECTION 3 — Project Paths and data loading
# ============================================================

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite"

GRAPH_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "aria_lite_graph_v2.pkl")

with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

print("Graph loaded")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Mounted at /content/drive
Graph loaded
Nodes: 1108
Edges: 19839


In [4]:
# ============================================================
# SECTION 4 — Helper: Entity Check and Query → Entity Mapping
# ============================================================

def is_entity(node):
    return G.nodes[node].get("type") == "entity"

def simple_entity_extract(query):
    return query.lower().split()

In [5]:
# ============================================================
# SECTION 5 — Helper: Entity Check
# ============================================================

def get_papers_from_entity(entity):

    papers = []

    if entity not in G:
        return papers

    for neighbor in G.neighbors(entity):

        if not is_entity(neighbor):
            papers.append(neighbor)

    return papers

In [6]:
# ============================================================
# SECTION 6 — 2-Hop Expansion
# ============================================================

def expand_from_entity(entity):

    result = {
        "direct_papers": set(),
        "related_entities": set()
    }

    if entity not in G:
        return result

    for n1 in G.neighbors(entity):

        # paper nodes
        if not is_entity(n1):
            result["direct_papers"].add(n1)

            # second hop
            for n2 in G.neighbors(n1):
                if is_entity(n2):
                    result["related_entities"].add(n2)

    return result

In [7]:
# ============================================================
# SECTION 7 — Main Graph Query Function
# ============================================================

from collections import Counter

def graph_query(query):

    # ----------------------------------------
    # Extract entities from query
    # ----------------------------------------

    entities = simple_entity_extract(query)

    # ----------------------------------------
    # Storage
    # ----------------------------------------

    paper_scores = Counter()
    all_entities = set()

    # ----------------------------------------
    # Traverse graph
    # ----------------------------------------

    for ent in entities:

        # skip if entity not in graph
        if ent not in G:
            continue

        expansion = expand_from_entity(ent)

        # store related entities
        all_entities.update(expansion["related_entities"])

        # ------------------------------------
        # Score papers
        # ------------------------------------

        # entity degree
        degree = G.degree(ent)

        # avoid divide by zero
        score = 1 / (degree + 1)

        for paper_id in expansion["direct_papers"]:

            # accumulate score
            paper_scores[paper_id] += score

    # ----------------------------------------
    # Rank papers
    # ----------------------------------------

    ranked_papers = sorted(
        paper_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # ----------------------------------------
    # Return results
    # ----------------------------------------

    return {
        "query": query,
        "entities_found": list(entities),
        "papers": ranked_papers,
        "related_entities": list(all_entities)
    }

In [8]:
# ============================================================
# SECTION 8 — Test Query
# ============================================================

result = graph_query("breast cancer HER2 immunotherapy")

print("Entities:", result["entities_found"])
print("\nPapers:", len(result["papers"]))
print("Related entities:", len(result["related_entities"]))

Entities: ['breast', 'cancer', 'her2', 'immunotherapy']

Papers: 236
Related entities: 484


In [9]:
# ============================================================
# SECTION 9 — Inspect Retrieved Papers
# ============================================================

def show_papers(ranked_papers, papers_data, top_k=5):

    # ----------------------------------------
    # Create PMID → Paper lookup
    # ----------------------------------------

    paper_lookup = {}

    for p in papers_data:
        paper_lookup[p["pmid"]] = p

    # ----------------------------------------
    # Display top results
    # ----------------------------------------

    for pmid, score in ranked_papers[:top_k]:

        if pmid not in paper_lookup:
            continue

        p = paper_lookup[pmid]

        print("\n" + "=" * 80)

        print("PMID:", pmid)
        print("Score:", round(score, 4))

        print("\nTitle:")
        print(p["title"])

        print("\nAbstract:")
        print(p["abstract_raw"])

In [10]:
# ============================================================
# SECTION 10 — Load Paper Metadata
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite"

PAPER_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "clean_papers.json")

with open(PAPER_PATH, "r") as f:
    papers_data = json.load(f)

In [11]:
# ============================================================
# SECTION 11 — Full Test Pipeline
# ============================================================

result = graph_query("breast cancer HER2 immunotherapy")

print("=== ENTITIES ===")
print(result["entities_found"])

print("\n=== SAMPLE PAPERS ===")
show_papers(result["papers"], papers_data)

=== ENTITIES ===
['breast', 'cancer', 'her2', 'immunotherapy']

=== SAMPLE PAPERS ===

PMID: 41889096
Score: 0.0098

Title:
Dual HER2/ERα Inhibitors for Breast and Ovarian Cancer: An Integrated Computational Study on 1,2,4-Oxadiazole Derivatives.

Abstract:
{'UNLABELLED': 'The 1,2,4-oxadiazole scaffold has attracted considerable interest as a privileged structure for anticancer drug development due to its favorable physicochemical properties and multimodal bioactivity. This study presents a comprehensive computational investigation to evaluate the potential of a series of 1,2,4-oxadiazole derivatives as dual inhibitors of the human epidermal growth factor receptor 2 (HER2) and estrogen receptor alpha (ERα), two key drivers in these malignancies. An integrated in silico strategy was employed, combining density functional theory (DFT), molecular docking and dynamics simulations, pharmacokinetic profiling, and machine learning models. Our workflow identified several lead compounds exhibit